### Endpoint for SAFE conversion process in DPR service that takes as input a path to a SAFE product in a bucket, calls CPM to transform the product, then returns the path to the converted product.

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-675


In [ ]:
# Init environment before running a demo notebook.
import os
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_cpm2()

In [ ]:
from rs_client.rs_client import RsClient
rs_server_href = os.getenv("RSPY_WEBSITE")
rs_server_api_key = os.environ.get("RSPY_APIKEY")
generic_client = RsClient(rs_server_href, rs_server_api_key, OWNER_ID, None)

In [ ]:
import os
import s3fs

safe_name = "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE"

fs = s3fs.S3FileSystem(
    key=os.environ["S3_ACCESSKEY"],
    secret=os.environ["S3_SECRETKEY"],
    client_kwargs={
        "endpoint_url": os.environ["S3_ENDPOINT"],
        "region_name": os.environ["S3_REGION"],
    },
)

fs.put(
    f"../../../CDSE/{safe_name}",
    f"rs-dev-cluster-temp/conversion/{safe_name}",
    recursive=True,
)



In [ ]:
safe_name = "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE"

result = submit_dask_cpm_task(f"""
def task():
    import os
    import shutil
    import fsspec
    from pathlib import Path
    from eopf.config import EOConfiguration
    from eopf.store.convert import convert

    safe_name = "{safe_name}"
    safe_s3 = f"rs-dev-cluster-temp/conversion/{{safe_name}}"
    local_safe = f"/tmp/{{safe_name}}"
    target = "/tmp/zarr-output-from-s3"

    fs = fsspec.filesystem(
        "s3",
        key=os.environ["S3_ACCESSKEY"],
        secret=os.environ["S3_SECRETKEY"],
        client_kwargs={{
            "endpoint_url": os.environ["S3_ENDPOINT"],
            "region_name": os.environ["S3_REGION"],
        }},
    )

    shutil.rmtree(local_safe, ignore_errors=True)
    shutil.rmtree(target, ignore_errors=True)

    fs.get(safe_s3, local_safe, recursive=True)

    cfg = EOConfiguration()
    cfg["dask_context__cluster_config__memory_limit"] = "24GiB"
    cfg["dask_context__cluster_config__n_workers"] = 1
    cfg["dask_context__cluster_config__threads_per_worker"] = 1
    cfg["dask_context__cluster_config__processes"] = True

    _, product_name = convert(local_safe, target, target_format="zarr")
    zarr_path = f"{{target}}/{{product_name}}.zarr"

    return {{
        "product_name": product_name,
        "local_safe_exists": Path(local_safe).exists(),
        "zarr_exists": Path(zarr_path).exists(),
        "zarr_path": zarr_path,
    }}
""")

result


In [ ]:
legacy_products = [
    #"S3A_OL_1_EFR____20240626T125108_20240626T125215_20240626T141905_0067_114_052_3780_PS1_O_NR_004.SEN3",
    #"S3A_OL_2_LFR____20240430T083943_20240430T084243_20240501T092030_0179_112_007_2160_PS1_O_NT_002.SEN3.zip",
    #"S3B_OL_2_LFR____20240430T130043_20240430T130343_20240501T004249_0179_092_252_1980_PS2_O_NT_002.SEN3",
    #"S3B_SY_2_V10____20240611T000000_20240620T235959_20240622T121945_EUROPE____________PS2_O_ST_002.SEN3",
    "S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.SAFE",
]

payloads = [
    {
        "input_safe_path": f"s3://rs-dev-cluster-temp/conversion/{legacy_product}",
        "output_zarr_dir_path": "s3://rs-dev-cluster-temp/conversion/",
    }
    for legacy_product in legacy_products
]

In [ ]:
dpr_client = generic_client.get_dpr_client()

# Launch all conversion jobs - "/dpr/processes/{resource}/execution" endpoint with "conv_safe_zarr" as resource
job_status_list = [dpr_client.run_conv_safe_zarr(payload, cluster_info_eopf) for payload in payloads]

results = [dpr_client.wait_for_job(job_status, logger=None, job_name="Conversion Processor") for job_status in job_status_list]
results

In [ ]:
responses = [requests.get(f"{dpr_client.href_service}/dpr/jobs/{job_status['jobID']}").json() for job_status in job_status_list]
responses

In [ ]:
# NOTE: this can only be run from the right venv kernel and by using: 
# from resources.dask_clusters.dask_venv import init_dask_cluster_cpm2_venv
# gateway, cluster, client = init_dask_cluster_cpm2_venv(
#     scale=1,
#     worker_cores=3,
#     worker_memory=12,
#     scheduler_memory_limit=60,
# )

# from distributed import get_worker

# def simple_task():
#     worker = get_worker()
#     return {
#         "worker": worker.address,
#         "message": "task ran on dask-cpm",
#     }

# future = client.submit(simple_task)
# future.result()
